# Inference

Runs the fine-tuned JuriBERT bi-encoder over the held-out test decisions to surface **implicit** article citations: preprocess the neutral test chunks, then FAISS-search the nearest Civil Code articles with triple explicit-citation filtering.

## Preprocess the test set

Builds a neutral evaluation set: keeps only test chunks that are long enough and contain **no** explicit legal keyword (`article`, `loi`, `code`), so the model is evaluated on implicit references only.

In [ ]:
import os

# Load the file
path = "artifacts/chunks/chunks_test.parquet"  # not shipped — regenerated by this step (see DATA.md)
df = pd.read_parquet(path)

# Conditions
long_enough = df["chunk_text"].str.len() >= 100
no_keywords = ~df["chunk_text"].str.contains(r"\b(article|articles|loi|code)\b", case=False, na=False)

# Apply both filters (long AND no keywords)
df_cleaned = df[long_enough & no_keywords]

# Statistics
total_chunks = len(df)
num_short = (~long_enough).sum()
num_with_keywords = (~no_keywords).sum()
num_cleaned = len(df_cleaned)

# Summary
print(f"Total initial chunks: {total_chunks:,}")
print(f"Chunks < 100 characters: {num_short:,}")
print(f"Chunks containing 'article|loi|code': {num_with_keywords:,}")
print(f"Retained chunks (long & no keywords): {num_cleaned:,}")

# Save
output_dir = "artifacts/test"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "chunks_test_clean.parquet")
df_cleaned.to_parquet(output_path, index=False)

print(f"File saved: {output_path}")

## Load the trained bi-encoder

Defines the custom SBERT modules (JuriBERT backbone + multi-head attention + MLP head) needed to reload the model trained in step 04.

In [ ]:
import os
import json

import torch
from torch import nn

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, models

# ========== custom modules: attention & TransformerWithAttention =============
class AttentionLayer(nn.Module):
    """Multi-head attention layer applied to a sequence of embeddings."""
    def __init__(self, input_dim: int, num_heads: int = 8):
        super().__init__()
        self.attention = nn.MultiheadAttention(input_dim, num_heads)
        self.input_dim, self.num_heads = input_dim, num_heads

    def forward(self, x):
        # x shape: (seq_len, batch_size, embedding_dim)
        # Self-attention recomputes each token based on all other tokens
        attn_out, _ = self.attention(x, x, x)
        return attn_out

    def save(self, path):
        torch.save(self.state_dict(), os.path.join(path, "attention.pt"))
        json.dump({"input_dim": self.input_dim, "num_heads": self.num_heads},
                  open(os.path.join(path, "attention_config.json"), "w"))

    @classmethod
    def load(cls, path):
        cfg = json.load(open(os.path.join(path, "attention_config.json")))
        obj = cls(cfg["input_dim"], cfg["num_heads"])
        obj.load_state_dict(torch.load(os.path.join(path, "attention.pt"), map_location="cpu"))
        return obj

class TransformerWithAttention(nn.Module):
    """Main backbone: JuriBERT + multi-head attention + linear projection."""
    def __init__(self, transformer, attention, tokenizer):
        super().__init__()
        self.transformer, self.attention, self.tokenizer = transformer, attention, tokenizer
        # Linear projection to transform each token embedding after attention
        self.projection = nn.Sequential(
            nn.Linear(transformer.config.hidden_size, transformer.config.hidden_size),
            nn.ReLU(),
        )

    def forward(self, features):
        ids, mask = features["input_ids"], features["attention_mask"]
        # Pass through JuriBERT -> last_hidden_state (batch, seq, dim)
        x = self.transformer(ids, attention_mask=mask).last_hidden_state
        # Permute to (seq, batch, dim) as required by PyTorch MultiheadAttention
        x = self.attention(x.permute(1, 0, 2)).permute(1, 0, 2)
        # Dense + ReLU projection on each token embedding
        features["token_embeddings"] = self.projection(x)
        return features

    def tokenize(self, texts, text_pair=None):
        if isinstance(texts, str): texts = [texts]
        if isinstance(text_pair, str): text_pair = [text_pair]
        return self.tokenizer(
            texts, text_pair, padding=True, truncation=True, max_length=512,
            return_tensors="pt"
        )

    def save(self, path):
        torch.save(self.state_dict(), os.path.join(path, "transformer_with_attention.pt"))
        self.tokenizer.save_pretrained(os.path.join(path, "juribert_tokenizer"))
        self.transformer.save_pretrained(os.path.join(path, "juribert_model"))
        self.attention.save(path)
        torch.save(self.projection.state_dict(), os.path.join(path, "projection.pt"))

    @classmethod
    def load(cls, path):
        trf = AutoModel.from_pretrained(os.path.join(path, "juribert_model"))
        tok = AutoTokenizer.from_pretrained(os.path.join(path, "juribert_tokenizer"))
        attn = AttentionLayer.load(path)
        obj = cls(trf, attn, tok)
        obj.projection.load_state_dict(torch.load(os.path.join(path, "projection.pt"), map_location="cpu"))
        obj.load_state_dict(torch.load(os.path.join(path, "transformer_with_attention.pt"), map_location="cpu"))
        return obj

class MLPHead(nn.Module):
    """Dense head (MLP) applied to the global embedding after pooling."""
    def __init__(self, input_dim: int, hidden_dim: int = 512, output_dim: int | None = None):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim or input_dim)
        )

    def forward(self, features, **_):
        features["sentence_embedding"] = self.mlp(features["sentence_embedding"])
        return features

    def save(self, path):
        torch.save(self.state_dict(), os.path.join(path, "mlp_head.pt"))
        cfg = {"input_dim": self.mlp[0].in_features,
               "hidden_dim": self.mlp[0].out_features,
               "output_dim": self.mlp[-1].out_features}
        json.dump(cfg, open(os.path.join(path, "mlp_head_config.json"), "w"))

    @classmethod
    def load(cls, path):
        cfg = json.load(open(os.path.join(path, "mlp_head_config.json")))
        obj = cls(cfg["input_dim"], cfg["hidden_dim"], cfg["output_dim"])
        obj.load_state_dict(torch.load(os.path.join(path, "mlp_head.pt"), map_location="cpu"))
        return obj


# JuriBERT bi-encoder inference pipeline (triple filtering)

## 0. Imports and initialization
Imports required Python libraries (text processing, embeddings, FAISS, pandas, etc.).

## 1. Paths and hyperparameters
Defines data paths (chunks, models, civil code...) and main hyperparameters (k, FAISS threshold...).

## 2. Article extraction and normalization
Provides functions for extracting explicitly cited articles (robust regex), handling typographic variants (hyphens, spaces), and normalizing text.

## 3. Loading and preparing data
Loads decisions (JSON), extracts motivations and full text, builds the FAISS index from civil code article embeddings.

## 4. Inference
For each chunk, finds the k nearest articles, then applies triple filtering (motivation + full text + fuzzy) to retain only implicit references.

## 5. Saving results
Exports predictions as Parquet and Excel files.

In [ ]:
# -*- coding: utf-8 -*-
"""
JuriBERT bi-encoder inference (triple filtering)

    Chunks:    parquet 'chunks_test_clean.parquet'
    Model:     'artifacts/model/bi_encoder_juribert'  # not shipped — regenerated by this step (see DATA.md)
    Decisions: Judilibre JSON under 'artifacts/raw_decisions'
    Articles:  Complete civil code (JSON)
    Output:    list of unique, filtered predictions

Python >= 3.10, sentence-transformers >= 3.0, transformers >= 4.40, faiss-cpu,
numpy, pandas, tqdm, nltk (punkt) required.
"""

# == 0. GENERAL IMPORTS =====================================================

import os
import re
import json
import unicodedata
import faiss
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
from typing import List, Dict, Tuple, Any, Set, Union, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import nltk

from sentence_transformers import SentenceTransformer, models

os.environ["TOKENIZERS_PARALLELISM"] = "false"

nltk.download("punkt", quiet=True)
from nltk.tokenize import sent_tokenize

# == 1. PATHS & HYPERPARAMETERS =============================================

ROOT = Path(".")

PARQUET_PATH = ROOT / "artifacts/test/chunks_test_clean.parquet"

MODEL_DIR = ROOT / "artifacts/model/bi_encoder_juribert"  # trained by 04_model_training; not shipped - see DATA.md

ARTICLES_JSON = ROOT / "DATA/inputs/civil_code_articles.json"
EQUIV_JSON = ROOT / "DATA/inputs/equivalences_new_to_old.json"

DECISIONS_DIR = ROOT / "artifacts/raw_decisions"  # 182k raw decisions; regenerated by step 01, not shipped - see DATA.md

K_NEIGHBORS = 5       # Number of neighbors to return (k-NN)
L2_THRESHOLD = 0.5738 # L2 squared threshold for filtering true matches

# == 2. ARTICLE EXTRACTION (regex + fuzzy) ===================================

# Unicode hyphens to normalize
HYPHENS = r"[\-\u2010-\u2015\u2212]"

def norm_text(s: str) -> str:
    """Normalize string: standardize hyphens, non-breaking spaces, unicode."""
    if not isinstance(s, str):
        return s
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(HYPHENS, "-", s)
    s = s.replace("\u00A0", " ")
    return s

# Main regex for detecting civil code article mentions
ARTICLE_REGEX = re.compile(
    rf"""
    \b(?:articles?|art\.)\s+
    (?P<num>
        \d{{1,4}}(?:{HYPHENS}\d+)?
        (?:\s*à\s*\d{{1,4}}(?:{HYPHENS}\d+)?)?
        (?:\s*(?:,|\bet\b)\s*
            \d{{1,4}}(?:{HYPHENS}\d+)?(?:\s*à\s*\d{{1,4}}(?:{HYPHENS}\d+)?)?
        )*
    )
    """,
    re.VERBOSE | re.IGNORECASE,
)

# Regex to find any isolated number (with optional hyphen suffix) in text
_NUM_RX = re.compile(rf"\b\d{{1,4}}(?:{HYPHENS}\d+)?\b")

# Regex for 4-digit numbers that could be written with a space ("1234" or "1 234")
_SPACE_RX = re.compile(r"^(\d)(\d{3})(-\d+)?$")

def _space_variant(num: str) -> Optional[str]:
    """Try to match '1234' as '1 234' format."""
    m = _SPACE_RX.match(num)
    return f"{m.group(1)} {m.group(2)}{m.group(3) or ''}" if m else None

def levenshtein(a: str, b: str) -> int:
    """Compute Levenshtein edit distance between two strings."""
    if len(a) < len(b):
        return levenshtein(b, a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            ins = prev[j] + 1
            delet = curr[j - 1] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, delet, sub))
        prev = curr
    return prev[-1]

def is_transposition(a: str, b: str) -> bool:
    """Check if a and b differ only by swapping two adjacent characters."""
    if len(a) != len(b) or a == b:
        return False
    diff = [i for i, (x, y) in enumerate(zip(a, b)) if x != y]
    return len(diff) == 2 and a[diff[0]] == b[diff[1]] and a[diff[1]] == b[diff[0]]

@lru_cache(maxsize=None)
def _basic_patterns(num: str, alt: Optional[str] = None) -> List[str]:
    """Return regex patterns for an article number and its variants."""
    pats = [fr"\b{re.escape(num)}\b"]
    if alt:
        pats.append(fr"\b{re.escape(alt)}\b")
    for v in filter(None, (_space_variant(num), _space_variant(alt) if alt else None)):
        pats.append(fr"\b{re.escape(v)}\b")
    return pats

@lru_cache(maxsize=None)
def _follow_patterns(num: str) -> List[str]:
    """Build patterns for 'art. N et suivants', including nearby article family."""
    base, sub = num.split("-", 1) if "-" in num else (num, None)
    def _mk(n: str) -> List[str]:
        prox = _space_variant(n)
        core = [n] + ([prox] if prox else [])
        prefix = r"(?:art\.?|articles?)?\s*"
        suffix = r"\s+et\s+suiv\w*\b"
        return [fr"\b{prefix}{re.escape(c)}{suffix}" for c in core]
    pats: List[str] = []
    if len(base) == 4:
        for d in range(1, 6):
            pats.extend(_mk(str(int(base) - d)))
    if sub is not None:
        pats.extend(_mk(f"{base}-{sub}"))
    pats.extend(_mk(base))
    return pats

def _mentioned(text: str, num: str, alt_map: Dict[str, str]) -> bool:
    """Check if num (or its variant) appears literally or with 'et suivants'."""
    alt = alt_map.get(num)
    for p in _basic_patterns(num, alt) + _follow_patterns(num):
        if re.search(p, text, flags=re.I):
            return True
    return False

def _mentioned_fuzzy(text: str, num: str, alt_map: Dict[str, str], max_edit: int = 1) -> bool:
    """Tolerant check: exact patterns first, then edit distance and transpositions."""
    if not isinstance(text, str):
        return False
    variants: Set[str] = {num}
    if "-" in num:
        variants.add(num.split("-")[0])
    alt = alt_map.get(num)
    if alt:
        variants.add(alt)
        if "-" in alt:
            variants.add(alt.split("-")[0])
    if any(_mentioned(text, v, {}) for v in variants):
        return True
    for cand in _NUM_RX.findall(text):
        if any(levenshtein(cand, v) <= max_edit or is_transposition(cand, v) for v in variants):
            return True
    return False

def extract_articles(text: str, valid: Set[str], max_edit_distance: int = 1) -> Set[str]:
    """Extract explicitly cited articles from text, handling lists, ranges, and typos."""
    raw: Set[str] = set()
    for m in ARTICLE_REGEX.finditer(text):
        nums = m.group("num")
        for part in re.split(r"\s*(?:,|\bet\b)\s*", nums):
            if "à" in part:
                a1, a2 = map(str.strip, part.split("à"))
                base1, base2 = a1.split("-")[0], a2.split("-")[0]
                if base1 == base2 or a2.startswith(base1):
                    cand = sorted([a for a in valid if a.startswith(base1)],
                                  key=lambda x: [int(p) for p in x.split("-")])
                    capturing = False
                    for art in cand:
                        if art == a1:
                            capturing = True
                        if capturing:
                            raw.add(art)
                        if art == a2:
                            break
                elif "-" not in a1 and "-" not in a2:
                    try:
                        for i in range(int(a1), int(a2) + 1):
                            art = str(i)
                            if art in valid:
                                raw.add(art)
                    except ValueError:
                        raw.update({a1, a2})
                else:
                    raw.update({a1, a2})
            else:
                raw.add(part)
    clean: Set[str] = set()
    digits_in_text = set(_NUM_RX.findall(text))
    for art in raw:
        if art not in valid:
            continue
        base = art.split("-")[0]
        erroneous = any(
            cited != art and (
                cited == base or
                levenshtein(cited, art) <= max_edit_distance or
                is_transposition(cited, art)
            )
            for cited in digits_in_text
        )
        if not erroneous:
            clean.add(art)
    return clean

In [ ]:
# 2. MOTIVATION AND FULL TEXT EXTRACTION

def extract_motivation(data: dict) -> str:
    """Extract the motivation section from a decision JSON."""
    zones = data.get("zones", {})
    if not zones.get("motivations"):
        return ""
    mot = zones["motivations"][0]
    s, e = mot.get("start"), mot.get("end")
    txt = data.get("text", "")
    return txt[s:e].strip() if s is not None and e and e <= len(txt) else ""

def load_motivations_and_full(json_dir: Path) -> (Dict[str, str], Dict[str, str]):
    """Load motivations and full text from all decision JSON files."""
    sentinel = "[DÉBATS NON PUBLICS – Motivation de la décision occultée]"
    mot_out = {}
    full_out = {}
    for p in tqdm(json_dir.rglob("*.json"), desc="parsing decisions"):
        try:
            data = json.loads(p.read_text("utf8"))
        except Exception:
            continue
        num = data.get("number", "").strip()
        date = data.get("decision_date", "").strip()
        if not num or not date:
            continue
        dec_id = f"{num}__{date}"
        mot = extract_motivation(data)
        full_txt = norm_text(data.get("text", ""))
        mot_out[dec_id] = norm_text(mot) if mot and sentinel not in mot else ""
        full_out[dec_id] = full_txt
    return mot_out, full_out

print("Loading motivations & full text...")
DEC2MOT, DEC2FULL = load_motivations_and_full(DECISIONS_DIR)

print("Extracting explicit citations...")
with ARTICLES_JSON.open() as f:
    ARTICLES = json.load(f)
VALID_ART = set(ARTICLES.keys())

# Extract explicitly cited articles from motivations
DEC2EXPL = {d: extract_articles(mot, VALID_ART) for d, mot in DEC2MOT.items()}
# Same for full text
DEC2EXPL_FULL = {d: extract_articles(full, VALID_ART) for d, full in DEC2FULL.items()}

with EQUIV_JSON.open() as f:
    ALT_MAP = {str(k): str(v) for k, v in json.load(f).items()}

# 3. BI-ENCODER & FAISS INDEX

# Load each module of the fine-tuned bi-encoder
backbone = TransformerWithAttention.load(MODEL_DIR / "0_TransformerWithAttention")
pooling = models.Pooling.load(MODEL_DIR / "1_Pooling")
mlp_head = MLPHead.load(MODEL_DIR / "2_MLPHead")
BI_ENCODER = SentenceTransformer(modules=[backbone, pooling, mlp_head])

# Get embedding dimension
DIM = BI_ENCODER.get_sentence_embedding_dimension()
# Encode all civil code articles for the FAISS index
ART_EMB = BI_ENCODER.encode(
    ["[ARTICLE] " + t for t in ARTICLES.values()],
    batch_size=64, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

faiss.omp_set_num_threads(1)
IDX = faiss.IndexFlatL2(DIM)
IDX.add(ART_EMB)
ID2ART = np.array(list(ARTICLES.keys()))

def search(v: np.ndarray, k: int = K_NEIGHBORS):
    """Query the FAISS index for k nearest neighbors."""
    q = np.ascontiguousarray(np.atleast_2d(v).astype("float32"))
    return IDX.search(q, k)

# 4. LOAD CHUNKS

df = pd.read_parquet(PARQUET_PATH).rename(columns={"chunk_text": "text_raw"})
print(f"{len(df):,} chunks loaded")

# 5. INFERENCE (triple filtering: motivation + full text)

def predict_for_decision(dec_id: str, chunk_rows: List[dict]) -> List[dict]:
    """Run inference for all chunks of a single decision with triple filtering."""
    mot_txt = DEC2MOT.get(dec_id, "")
    full_txt = DEC2FULL.get(dec_id, "")
    explicit_mot = DEC2EXPL.get(dec_id, set())
    explicit_full = DEC2EXPL_FULL.get(dec_id, set())
    blocked_bases = {a.split("-")[0] for a in (explicit_mot | explicit_full)}

    def _base_mentioned(text: str, base: str) -> bool:
        if not text:
            return False
        pat = rf"\b(?:art\.?|articles?)?\s*{re.escape(base)}(?:-\d+)?\b"
        return re.search(pat, text, flags=re.I) is not None

    batch_txts = ["[DECISION] " + r["text_raw"] for r in chunk_rows]
    embs = BI_ENCODER.encode(
        batch_txts, batch_size=32, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")

    D, I = search(embs)
    preds = []
    for rec, idxs, dists in zip(chunk_rows, I, D):
        for idx, dist in zip(idxs, dists):
            if dist > L2_THRESHOLD:
                break
            art = ID2ART[int(idx)]
            art_base = art.split("-")[0]

            # Triple filtering: exclude articles already explicitly cited
            if (
                art in explicit_mot or
                art in explicit_full or
                art_base in blocked_bases or
                _base_mentioned(mot_txt, art_base) or
                _base_mentioned(full_txt, art_base) or
                _mentioned_fuzzy(mot_txt, art, ALT_MAP) or
                _mentioned_fuzzy(full_txt, art, ALT_MAP)
            ):
                continue

            preds.append({
                "decision_id": dec_id,
                "chunk_id": rec["chunk_id"],
                "pred_art": art,
                "distance": float(dist),
                "text": rec["text_raw"],
            })
    return preds

# 6. MAIN LOOP

all_preds = []
for dec_id, subdf in tqdm(df.groupby("decision_id"), total=df["decision_id"].nunique(), desc="Decisions"):
    if dec_id not in DEC2MOT:
        continue
    all_preds.extend(predict_for_decision(dec_id, subdf.to_dict("records")))

print(f"{len(all_preds):,} chunk-article pairs (dist <= {L2_THRESHOLD})")

# Deduplication: keep only one prediction per (decision, chunk, article)
seen, uniq = set(), []
for p in sorted(all_preds, key=lambda x: x["distance"]):
    key = (p["decision_id"], p["chunk_id"], p["pred_art"])
    if key not in seen:
        uniq.append(p); seen.add(key)

print(f"{len(uniq):,} unique predictions after deduplication")

# Enrich each prediction with the corresponding article text
for p in uniq:
    p["article_text"] = ARTICLES.get(p["pred_art"], "[text not found]")

## Save results

In [ ]:
# Convert results to DataFrame
import os
os.makedirs("artifacts/inference", exist_ok=True)  # not shipped — regenerated by this step (see DATA.md)
df_preds = pd.DataFrame(uniq)

# Save as Parquet
parquet_output_path = ROOT / "artifacts/inference/output_predictions.parquet"
df_preds.to_parquet(parquet_output_path, index=False)
print(f"Results saved as Parquet at {parquet_output_path}")

# Save as Excel
excel_output_path = ROOT / "artifacts/inference/output_predictions.xlsx"
df_preds.to_excel(excel_output_path, index=False)
print(f"Results saved as Excel at {excel_output_path}")

## Duplicate removal and unique article count
- Remove duplicates (same chunk and predicted article, even across different decisions)
- Count the number of unique articles present in the predictions.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

# Load the predictions parquet
df = pd.read_parquet(ROOT / "artifacts/inference/output_predictions.parquet")  # not shipped — regenerated by this step (see DATA.md)

# Remove duplicates based on "text" and "pred_art"
df_unique = df.drop_duplicates(subset=["text", "pred_art"])
print(f"Shape after deduplication: {df_unique.shape}")

# Save deduplicated results
df_unique.to_parquet(ROOT / "artifacts/inference/output_predictions_unique.parquet", index=False)
df_unique.to_excel(ROOT / "artifacts/inference/output_predictions_unique.xlsx", index=False)